In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!curl -L "https://app.roboflow.com/ds/mMrqzZdf5N?key=GIj6LOZ9bo" > roboflow.zip

!unzip roboflow.zip

!rm roboflow.zip

In [ ]:
!ls /content/train/

In [ ]:
import tensorflow as tf

dataset_path = "/content/train"

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

In [ ]:
class_names = train_ds.class_names

print(class_names[:20])
print("Tổng số class:", len(class_names))

In [ ]:
!wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz

In [ ]:
!tar -xzf food-101.tar.gz

In [ ]:

!rm food-101.tar.gz

In [ ]:
!ls

In [ ]:
!cp -r /content/train/* /content/food-101/images/

In [ ]:
!ls /content/food-101/images/

In [ ]:
import os
import shutil

# Đường dẫn tới thư mục chứa tất cả các món ăn
base_dir = '/content/food-101/images/'

print("Bắt đầu dọn dẹp và chuẩn hóa dữ liệu...")

for folder_name in os.listdir(base_dir):
    old_path = os.path.join(base_dir, folder_name)

    # Chỉ xử lý nếu nó là thư mục
    if os.path.isdir(old_path):
        # 1. Đưa về chữ thường và thay dấu cách bằng gạch dưới
        new_name = folder_name.lower().replace(' ', '_')
        new_path = os.path.join(base_dir, new_name)

        # Nếu tên cần thay đổi
        if old_path != new_path:
            # 2. Nếu thư mục đích ĐÃ TỒN TẠI (Ví dụ: 'pho' đã có, giờ đang xử lý 'Pho')
            if os.path.exists(new_path):
                print(f"Đang gộp: [{folder_name}] vào [{new_name}]")
                # Chuyển từng ảnh sang nhà mới
                for file_name in os.listdir(old_path):
                    shutil.move(os.path.join(old_path, file_name), os.path.join(new_path, file_name))
                # Xóa thư mục cũ trống rỗng
                os.rmdir(old_path)
            # 3. Nếu thư mục đích chưa tồn tại, chỉ cần đổi tên
            else:
                print(f"Đổi tên: [{folder_name}] -> [{new_name}]")
                os.rename(old_path, new_path)

print("\n HOÀN TẤT! Dữ liệu đã chuẩn 100%.")

In [ ]:
!ls /content/food-101/images/

In [ ]:
import os

save_dir = "/content/drive/MyDrive/datasets_mon_an"

os.makedirs(save_dir, exist_ok=True)

print("Đã tạo thư mục:", save_dir)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# 1. Dọn dẹp RAM trước khi chạy mẻ lớn
tf.keras.backend.clear_session()

# 2. Trỏ vào thư mục đã gộp (134 món)
dataset_path = "/content/food-101/images"

# 3. Load Dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_ds.class_names
print(f"🎯 Đã khóa mục tiêu: {len(class_names)} món ăn!")

# 4. Tính toán Trọng số (Class Weights)
print("⚖️ Đang tính toán trọng số cân bằng dữ liệu...")
train_labels = np.concatenate([y for x, y in train_ds], axis=0)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_dict = dict(enumerate(class_weights_array))
print("Tạo trọng số thành công!")

# Tối ưu hóa đọc dữ liệu
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# 5. Xây dựng Kiến trúc (Unified Model)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

# 6. Compile và Huấn luyện (Learning Rate an toàn 1e-4)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("🚀 Bắt đầu quá trình huấn luyện Tối thượng (10 Epochs)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights_dict
)

# 7. Lưu file Keras về Google Drive
save_path = "/content/drive/MyDrive/datasets_mon_an/unified_food_model.keras"
model.save(save_path)
print(f"Đã lưu thành công mô hình hợp nhất tại: {save_path}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("BẮT ĐẦU FINE-TUNING")

# 1. Nạp lại mô hình 63% ban đầu
model_path = "/content/drive/MyDrive/datasets_mon_an/unified_food_model.keras"
model = tf.keras.models.load_model(model_path)

# Trích xuất riêng phần mạng EfficientNet (Lớp thứ 2 trong Sequential, sau Data Augmentation)
base_model = model.layers[1]

# 2. Chiến thuật Mở khóa tinh vi:
# Chỉ mở khóa 30 lớp cuối cùng của EfficientNet để học các chi tiết hẹp gọn
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Đặc biệt: Đóng băng TOÀN BỘ các lớp BatchNormalization để chống nhiễu
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Đã cấu hình mở khóa an toàn!")

# 3. Compile lại với tốc độ học an toàn hơn (5e-5)
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

# 4. Bắt đầu luyện lại
print("Đang tiến hành Fine-Tuning.")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
    class_weight=class_weights_dict
)

model.save("/content/drive/MyDrive/datasets_mon_an/unified_food_model_v3.keras")
print("Đã lưu phiên bản V2!")